# Material de aula - FACENS - Aprendizado de Máquina


## Perceptron e Regra de Aprendizado de Hebb - Laboratório 03

In [18]:
#CÉLULA PERCEPTRON-LIB-01
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

In [19]:
#CÉLULA PERCEPTRON-LIB-02
#Função para carregamento de dados de treinamento
def loadDataFile(filename) :
    '''
    Esta função faz o carregamento de dados de um arquivo separado por virgulas (csv)
    
    Parametros
    -----------
    filename -> nome do arquivo de dados em formato csv
    
    
    Retorno
    -----------  
    dataframe contendo os dados do arquivo formatados numericamente.
    '''
    
    dfFile = pd.read_csv(filename, sep=';')
    
    #Os numeros estao com virgula, troque para ponto
    dfFile = dfFile.replace(regex=',', value='.')
    
    #print(dfFile.columns)
    
    #Converta para numeros
    #Verifique se a coluna existe antes de manipulá-la    
    if ('x1' in dfFile.columns) :
        dfFile['x1']  = pd.to_numeric(dfFile['x1'])

    if ('x2' in dfFile.columns) :
        dfFile['x2']  = pd.to_numeric(dfFile['x2'])

    if ('x3' in dfFile.columns) :
        dfFile['x3']  = pd.to_numeric(dfFile['x3'])

    if ('d' in dfFile.columns) :
        dfFile['d']  = pd.to_numeric(dfFile['d'])
    
    #print(dfFile.dtypes)
    
    return dfFile


dfTrainingFile = loadDataFile('Lab01_training.csv')
print(dfTrainingFile.dtypes)

dfOperationFile = loadDataFile('Lab01_operation.csv')
print(dfOperationFile.dtypes)

Amostra      int64
x1         float64
x2         float64
x3         float64
d            int64
dtype: object
x1    float64
x2    float64
x3    float64
dtype: object


In [20]:
dfTrainingFile.head()

,Amostra,x1,x2,x3,d
0,1,-0.6508,0.1097,4.0009,-1
1,2,-1.4492,0.8896,4.4005,-1
2,3,2.0850,0.6876,12.0710,-1
3,4,0.2626,1.1476,7.7985,1
4,5,0.6418,1.0234,7.0427,1


In [21]:
dfOperationFile

,x1,x2,x3
0,-0.3565,0.0620,5.9891
1,-0.7842,1.1267,5.5912
2,0.3012,0.5611,5.8234
3,0.7757,1.0648,8.0677
4,0.1570,0.8028,6.3040
5,-0.7014,1.0316,3.6005
6,0.3748,0.1536,6.1537
7,-0.6920,0.9404,4.4058
8,-1.3970,0.7141,4.9263
9,-1.8842,-0.2805,1.2548


In [22]:
#CÉLULA PERCEPTRON-LIB-03
# Função para cálculo de normalização de uma amostra
def tales(x1, x1_min, x1_max, x2_min, x2_max) :
    '''
    Esta função faz o escalamento de 1 amostra x1, que pertence ao intervalo [x1_min, x1_max], 
    dentro do intervalo [x2_min, x2_max]
    
    Parametros
    -----------
    x1     -> valor a ser normalizado
    x1_min -> valor minimo atingido por x1
    x1_max -> valor maximo atingido por x1
    x2_min -> valor minimo da escala de destino
    x2_max -> valor maximo da escala de destino    
    
    Retorno
    -----------  
    valor de x1, que pertence ao intervalo [x1_min, x1_max], projetado dentro do intervalo [x2_min, x2_max]
    '''
   
    #Teorema de Tales
    # ( x2 - x2_min ) / ( x2_max - x2_min ) = ( x1 - x1_min ) / (x1_max - % x1_min);
    
    #Definindo x1 como a escala do vetor dado e x2 como a escala de saída (variavel de interesse), isola-se x2:
    x2 = ( ( x1 - x1_min ) / (x1_max - x1_min) )* ( x2_max - x2_min ) + x2_min
    
    return x2

In [23]:
#CÉLULA PERCEPTRON-LIB-04
# Função para cálculo de normalização de um vetor de amostras
def scale(x2_min, x2_max, sampleArray) :
    '''
    Esta função faz o escalamento de 1 vetor de amostras para dentro do intervalo [x2_min, x2_max]
    Ela retorna um vetor com os dados normalizados dentro do dominio dado
    
    Parametros
    -----------
    x2_min      -> valor minimo da escala de destino
    x2_max      -> valor maximo da escala de destino
    sampleArray -> vetor a ser normalizado   
    
    Retorno
    -----------  
    vetor sampleArray projetado dentro do intervalo [x2_min, x2_max]   
    '''
    
    x_max = max(sampleArray)
    x_min = min(sampleArray)
    
    sampleArrayNorm = [tales(x, x_min, x_max, x2_min, x2_max) for x in sampleArray]
    
    return sampleArrayNorm

In [24]:
#Normalizando array de treino

dfTrainingFile = loadDataFile('Lab01_training.csv')
dfTrainingFile
dfTrainingFileNorm = dfTrainingFile
dfTrainingFileNorm['x1'] = scale(-1,1, dfTrainingFile['x1'])
dfTrainingFileNorm['x2'] = scale(-1,1, dfTrainingFile['x2'])
dfTrainingFileNorm['x3'] = scale(-1,1, dfTrainingFile['x3'])

dfTrainingFile.head()

,Amostra,x1,x2,x3,d
0,1,-0.590291,-0.543460,-0.560073,-1
1,2,-1.000000,0.616328,-0.482824,-1
2,3,0.813619,0.315934,1.000000,-1
3,4,-0.121568,1.000000,0.174061,1
4,5,0.073023,0.815302,0.027953,1


In [25]:
#Normalizando array de teste

dfTestFile = loadDataFile('Lab01_operation.csv')
dfTestFile
dfTestFileNorm = dfTestFile
dfTestFileNorm['x1'] = scale(-1,1, dfTestFileNorm['x1'])
dfTestFileNorm['x2'] = scale(-1,1, dfTestFileNorm['x2'])
dfTestFileNorm['x3'] = scale(-1,1, dfTestFileNorm['x3'])

dfTestFileNorm

,x1,x2,x3
0,0.148690,-0.513218,0.389805
1,-0.172901,1.000000,0.272997
2,0.643220,0.196134,0.341162
3,1.000000,0.912024,1.000000
4,0.534795,0.539653,0.482247
5,-0.110643,0.864838,-0.311395
6,0.698560,-0.383030,0.438125
7,-0.103575,0.735219,-0.074990
8,-0.633670,0.413587,0.077808
9,-1.000000,-1.000000,-1.000000


# 1 - Implementação do algoritmo de treinamento e ajuste de pesos do perceptron

Utilizando Jupyter e NumPy: para o problema apresentado, construa uma função para o cálculo do ajuste de pesos do perceptron.

Assinatura: **def perceptron_train(eta, vInputs, vOutputs, max_epochs) :**

### esta função faz o ajuste dos pesos do perceptron, com a taxa de aprendizado dada, juntamente com o vetor de amostras e de saídas desejadas dentro do limite de treinamento pre-estabelecido.
    
**Retorno da Função:** Retorna uma tupla contendo os pesos ajustados e a quantidade de épocas executadas para o treinamento


In [26]:
#CÉLULA PERCEPTRON-LIB-05
def perceptron_train(eta, vInputs, vOutputs, max_epochs) :    
    '''
    Algoritmo de treinamento do Perceptron Simples
    
    Parametros
    -----------
    eta        -> taxa de aprendizado
    inputs     -> vetor/matriz com as entradas
    outputs    -> vetor/matriz com as saidas
    max_epochs -> criterio de parada por epocas de treinamento
    
    Retorno
    -------
    Este método retorna uma tupla contendo o vetor de pesos ajustado e o numero de epocas executadas no treinamento.
    '''
    
    #IMPLEMENTE O CÓDIGO AQUI
    
    #Determine numero de entradas do perceptron e o numero de amostras   
    nInputs = vInputs.shape[0]#são 4 linhas
    nSamples = vInputs.shape[1]#são 3 colunas
        
    #Inicialize o vetor de pesos com valores aleatórios
    vWeights = np.random.rand(nInputs,1)
    
    #Monte o loop de cálculo do erro e ajuste dos pesos
    bError = True
    
    #while(bError == True):
    for epochCounter in range(max_epochs):
        
        if(bError == False):
            break
        
        bError = False
        for k in range(0,nSamples):
            u = np.matmul(vWeights.T,vInputs[:,[k]])#calcula o u e deita o array
            y = np.sign(u)
            
            if(y == vOutputs[k]):
                bError == True   
                vWeights += eta*(vOutputs[k] - y) * vInputs[:,[k]]
    
    return (vWeights, epochCounter-1)

# 2 - Implementação do algoritmo de operação do perceptron

Utilizando Jupyter e NumPy: para o problema apresentado, construa uma função para o cálculo de saída do perceptron.

Assinatura: **def perceptron_operation(vWeights, vInputs) :**

### esta função faz o cálculo do valor de saída do utilizando valores de pesos para um perceptron pré-ajustado.
    
**Retorno da Função:** Retorna um escalar com o valor de saída, no caso -1 ou 1

In [27]:
#CÉLULA PERCEPTRON-LIB-06

def perceptron_operation(vWeights, vInputs) :
    '''Algoritmo de operação do Perceptron Simples
    
    Parametros
    -----------
    vWeights : vetor/matriz com os pesos
    inputs   : vetor/matriz com as entradas
    
    Retorno
    -------
    Este método retorna o valor de saída do perceptron calculado segundo sua função de ativação signal(.)    
    ''' 

    u = np.matmul(vWeights.T,vInputs)#T = matriz transposta
    y = np.sign(u)
        
    return np.asscalar(y) #converte o numpy array 1x1 em um unico valor escalar.

# 3 - Implementação do algoritmo de execução do treinamento e operação do perceptron

### A partir da análise de um processo de destilação fracionada de petróleo observou-se que determinado óleo poderia ser classificado em duas classes de pureza {C1 e C2}, mediante a medição de três grandezas {x1, x2 e x3} que representam algumas das propriedades físico-químicas do óleo. 

### A equipe de engenheiros e cientistas pretende utilizar um perceptron para executar a classificação automática destas duas classes.

### Assim, baseadas nas informações coletadas do processo, formou-se o conjunto de treinamento em anexo, tomando por convenção o valor –1 para óleo pertencente à classe C1 e o valor +1 para óleo pertencente à classe C2. 

### Portanto, o neurônio constituinte do perceptron terá três entradas e uma saída, conforme a figura abaixo:

![image.png](attachment:image.png)

### Utilizando o algoritmo supervisionado de Hebb (regra de Hebb) para classificação de padrões, e assumindo-se a taxa de aprendizagem igual a 0.01, faça as seguintes atividades:

## 3.1 - Implemente o código que realiza o treinamento do perceptron para o problema dado.

## PERCEPTRON - inicio

In [28]:
#CÉLULA PERCEPTRON-MAIN-01

#IMPLEMENTE O CÓDIGO AQUI

#Parâmetros
eta = 0.01
nInputs = 3
max_epochs = 1000 

#Carrega dados - Treino
dfTrainingFile = loadDataFile('Lab01_training.csv')
dfTestFileNorm = dfTrainingFile.copy()

dfTrainingFileNorm['x1'] = scale(-1,1,dfTrainingFile['x1'])
dfTrainingFileNorm['x2'] = scale(-1,1,dfTrainingFile['x2'])
dfTrainingFileNorm['x3'] = scale(-1,1,dfTrainingFile['x3'])

nSamples = len(dfTrainingFileNorm['x1'])

col0 = np.ones((nSamples,1)) * (-1)
col1 = dfTrainingFileNorm['x1'].values.reshape(nSamples,1)
col2 = dfTrainingFileNorm['x2'].values.reshape(nSamples,1)
col3 = dfTrainingFileNorm['x3'].values.reshape(nSamples,1)

vInputs = np.array(col0)
vInputs = np.append(vInputs,col1,axis=1)
vInputs = np.append(vInputs,col2,axis=1)
vInputs = np.append(vInputs,col3,axis=1)

vOutputs = dfTrainingFileNorm['d'].values.reshape(nSamples, 1)


#Executa o treinamento
(vWeights,nEpochs) = perceptron_train(eta, vInputs.T, vOutputs, max_epochs)

#Verifica o resultado do treinamento (use as variáveis abaixo)
print(nEpochs)
print(vWeights)

0
[[0.23135643]
 [0.74420795]
 [0.69319867]
 [0.29475557]]


## 3.2 - Implemente o código que valida o resultado do treinamento

In [29]:
#Verifica resultados 

y=[]
for k in range(nSamples):
    y.append(perceptron_operation(vWeights, vInputs[[k], :].T))

dfValidation = dfTrainingFile.copy()
dfValidation['y'] = y

results = dfValidation['y'] == dfValidation['d']
accuracy = sum(results) / len(results) * 100

#Execute a operação para cada amostra de treinamento


#Verifique a precisão do treinamento


print('Taxa de acerto do conjunto de treinamento: {}%'.format(accuracy))

Taxa de acerto do conjunto de treinamento: 60.0%


## 3.3 - Implemente o código que executa o perceptron treinado com dados de operação

In [ ]:
#IMPLEMENTE O CÓDIGO AQUI

#Carrega dados - Teste


#Execute a operação para cada amostra de treinamento


#Exibe a saída
print(y)